# PenG — AI Học Tập
Clone, install, and test PenG from Google Colab with GPU T4.

**Prerequisites:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Clone repo
!git clone https://github.com/canhcutlo/PenG.git
%cd PenG

In [ ]:
# 2. Check GPU
!nvidia-smi

In [ ]:
# 3. Install dependencies
# Pillow>=10.0,<11 for Surya-ocr compatibility on Python 3.12
!pip install -r requirements-colab.txt
!pip install pyngrok nest-asyncio

In [ ]:
# 4. Compile check
!python -m compileall app

In [ ]:
# 5. Run unit tests (skip AI model integration tests)
!pytest tests/ -v -m "not integration"

In [ ]:
# 6. Start FastAPI server in background (& runs it in a subprocess)
#    Colab already has a running event loop so uvicorn.run() would crash.
#    Using shell uvicorn with & avoids that problem.
!uvicorn app.main:app --host 0.0.0.0 --port 8000 &

In [ ]:
# 7. Open public URL via ngrok
#    Get a free token at https://dashboard.ngrok.com
import time
from pyngrok import ngrok

# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
time.sleep(2)  # wait for uvicorn to start
public_url = ngrok.connect(8000)
print(f"\n🌐 Public URL: {public_url}")
print(f"📖 API docs:   {public_url}/docs")
print(f"🖥️ Frontend:   {public_url}")
print(f"❤️ Health:     {public_url}/api/health")

## Verification checklist

Mở `public_url` in trình duyệt và kiểm tra từng mục:

- [ ] **Health**: `/api/health` → `{"status":"ok","db":"ok"}`
- [ ] **Frontend**: `/` → hiện 5 tabs (Upload / Hỏi đáp / Quiz / Mindmap / Lịch sử)
- [ ] **Upload**: Kéo-thả file ảnh nhỏ → trả `doc_id` + `job_id`; poll `/api/jobs/{job_id}`
- [ ] **Query**: Nhập câu hỏi vào tab Hỏi đáp → trả answer
- [ ] **Mindmap**: Nhập `doc_id` vào tab Mindmap → hiển thị markmap
- [ ] **Quiz**: Generate quiz từ `doc_id` → hiển thị câu hỏi → submit → chấm điểm
- [ ] **History**: Tab Lịch sử hiển thị hoạt động đã log

## Troubleshooting

| Vấn đề | Cách sửa |
|---|---|
| `ModuleNotFoundError: surya` | Colab Python 3.12: chạy `!pip install "Pillow>=10.0,<11" && !pip install surya-ocr` |
| CUDA out of memory | Tải Qwen/Llama với 4-bit quantization; hoặc dùng CPU |
| ngrok không kết nối | Kiểm tra token tại https://dashboard.ngrok.com |
| Server không khởi động | Chạy lại cell 6, đợi 3-5 giây |